# GFS 2 m temperature — NWP quickstart

Fetch the latest GFS run's 2 m temperature analysis over a small bbox
and read the resulting Cloud-Optimized GeoTIFF.

**Requirements** (live download):

```bash
pip install earthlens[nwp]
conda install -c conda-forge eccodes libgdal-grib   # binary libs
```

`herbie-data` downloads only the `.idx` byte range for the requested
variable (>99 % bandwidth saving); `pyramids.grib.open_grib` reads it
with GDAL's GRIB driver, and earthlens crops it to the bbox.

In [ ]:
import datetime as dt

from earthlens.earthlens import EarthLens

# A recent 00Z cycle (GFS keeps ~10 days on NODD).
cycle_day = (dt.datetime.now(dt.UTC) - dt.timedelta(days=1)).strftime("%Y-%m-%d")
cycle_day

In [ ]:
lens = EarthLens(
    data_source="nwp",
    variables={"gfs": ["temperature_2m"]},
    start=cycle_day,
    end=cycle_day,
    lat_lim=[40, 45],
    lon_lim=[-80, -75],
    path="out/gfs",
    steps=[0],
    mirror="aws",
)
paths = lens.download(progress_bar=False)
paths

In [ ]:
from pyramids.grib import open_grib

# The output is a standard COG — open it with pyramids and inspect the grid.
ds = open_grib(str(paths[0]))
ds.shape

To request several lead times, pass `steps=[0, 6, 12, 24]` (or
`horizon=48`); each `(cycle, step)` yields one COG. Add
`aggregate=AggregationConfig(freq="1D", op="mean")` to reduce the stack
to daily means.